# Download assets: BioCLIP + BIRDS 525

Fetches the two things training needs and puts them where `data_utils.py` expects:

| Asset | Destination |
|---|---|
| BioCLIP ViT-B/16 checkpoint | `$LFCBM_MODELS_DIR/bioclip/open_clip_pytorch_model.bin` |
| BIRDS 525 images (ImageFolder) | `$LFCBM_DATA_ROOT/birds525/{train,val,test}/<CLASS>/` |
| Class list | `$LFCBM_DATA_ROOT/birds525.txt` |

Every cell is idempotent — re-running skips work that is already done.

**Prerequisites**
- `pip install huggingface_hub open_clip_torch kaggle torchvision`
- A Kaggle API token at `~/.kaggle/kaggle.json` (Kaggle → Settings → Create New Token), then `chmod 600 ~/.kaggle/kaggle.json`

Run this notebook from the repo root.

## 0. Paths

In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_ROOT   = Path.cwd()
MODELS_DIR  = Path(os.environ.get("LFCBM_MODELS_DIR", "/workspace/models"))
DATA_DIR    = Path(os.environ.get("LFCBM_DATA_ROOT", REPO_ROOT / "data"))

BIOCLIP_DIR = MODELS_DIR / "bioclip"
BIRDS_RAW   = DATA_DIR / "birds525_raw"    # kaggle unzips here, deleted at the end
BIRDS_DIR   = DATA_DIR / "birds525"        # final ImageFolder root

for p in (MODELS_DIR, DATA_DIR):
    p.mkdir(parents=True, exist_ok=True)

print(f"repo root   {REPO_ROOT}")
print(f"models dir  {MODELS_DIR}")
print(f"data dir    {DATA_DIR}")
print()
print("NOTE: if MODELS_DIR is not /workspace/models, export LFCBM_BIOCLIP_CKPT to point at")
print("      the checkpoint before training. That covers the backbone; the BioCLIP *concept")
print("      encoder* in utils.py still hardcodes /workspace/models/bioclip/.")

## 1. BioCLIP checkpoint

[`imageomics/bioclip`](https://huggingface.co/imageomics/bioclip) — a ViT-B/16 OpenCLIP model
trained on the TreeOfLife-10M dataset. ~600 MB.

In [ ]:
from huggingface_hub import hf_hub_download

BIOCLIP_REPO = "imageomics/bioclip"
BIOCLIP_DIR.mkdir(parents=True, exist_ok=True)

for fname in ("open_clip_pytorch_model.bin", "open_clip_config.json"):
    dest = BIOCLIP_DIR / fname
    if dest.exists():
        print(f"skip  {fname}  ({dest.stat().st_size / 1e6:.0f} MB, already present)")
        continue
    print(f"downloading {fname} ...")
    hf_hub_download(repo_id=BIOCLIP_REPO, filename=fname, local_dir=str(BIOCLIP_DIR))
    print(f"  -> {dest}  ({dest.stat().st_size / 1e6:.0f} MB)")

### Verify it loads, and check both feature taps

This also answers *what shape do I actually get* from `encode_image` vs a hook on
`visual.ln_post` — the two candidate backbone feature layers.

In [ ]:
import torch, open_clip

model, _, preprocess = open_clip.create_model_and_transforms("ViT-B-16")

ckpt = torch.load(BIOCLIP_DIR / "open_clip_pytorch_model.bin",
                  map_location="cpu", weights_only=False)
state_dict = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
state_dict = {k.replace("module.", "", 1) if k.startswith("module.") else k: v
              for k, v in state_dict.items()}

missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f"missing keys:    {len(missing)}  {missing[:3]}")
print(f"unexpected keys: {len(unexpected)}  {unexpected[:3]}")
model.eval()

x = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    print(f"\nencode_image(x)          -> {tuple(model.encode_image(x).shape)}   (after visual.proj)")

    captured = {}
    h = model.visual.ln_post.register_forward_hook(lambda m, i, o: captured.__setitem__("f", o))
    model.visual(x)
    h.remove()
    print(f"hook on visual.ln_post   -> {tuple(captured['f'].shape)}   (before visual.proj)")
    print(f"\nvisual.proj: {tuple(model.visual.proj.shape)}  <- the linear map between the two")

## 2. BIRDS 525 dataset

Kaggle [`gpiosenka/100-bird-species`](https://www.kaggle.com/datasets/gpiosenka/100-bird-species).
~2 GB, ~90k images, already split into `train/ valid/ test/` in ImageFolder layout.

In [ ]:
KAGGLE_DATASET = "gpiosenka/100-bird-species"

if BIRDS_DIR.exists() and any(BIRDS_DIR.iterdir()):
    print(f"skip download: {BIRDS_DIR} already populated")
elif BIRDS_RAW.exists() and any(BIRDS_RAW.iterdir()):
    print(f"skip download: {BIRDS_RAW} already populated (reorganize in the next cell)")
else:
    token = Path.home() / ".kaggle" / "kaggle.json"
    assert token.exists(), f"Kaggle API token not found at {token}"
    BIRDS_RAW.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
         "-p", str(BIRDS_RAW), "--unzip"],
        check=True,
    )
    print(f"downloaded to {BIRDS_RAW}")

### Reorganize into the layout `DATASET_ROOTS` expects

Kaggle names the validation split `valid/`; `data_utils.DATASET_ROOTS["birds525_val"]`
points at `val/`. This renames it and drops the non-image extras that ship with the zip
(`birds.csv`, a bundled `.h5` model).

In [ ]:
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}
BIRDS_DIR.mkdir(parents=True, exist_ok=True)

for src_name, dst_name in SPLIT_MAP.items():
    src, dst = BIRDS_RAW / src_name, BIRDS_DIR / dst_name
    if dst.exists():
        print(f"skip  {dst_name:6s} (already at {dst})")
        continue
    if not src.exists():
        raise FileNotFoundError(f"expected {src} from the Kaggle zip -- check the download")
    shutil.move(str(src), str(dst))
    n_cls = sum(1 for d in dst.iterdir() if d.is_dir())
    print(f"move  {src_name:6s} -> {dst_name:6s} ({n_cls} classes)")

# leftover extras (birds.csv, EfficientNet .h5, stray dirs) are not needed for training
if BIRDS_RAW.exists():
    leftovers = sorted(p.name for p in BIRDS_RAW.iterdir())
    print(f"\nremoving unused extras from {BIRDS_RAW}: {leftovers}")
    shutil.rmtree(BIRDS_RAW)

## 3. Class list

`train_cbm.py` reads `data_utils.LABEL_FILES[dataset]` and uses `len(classes)` as the
output dimension of the final layer, so the file must have **exactly one line per class,
in `torchvision.datasets.ImageFolder` order (sorted directory names) and no trailing
newline** — a trailing newline yields a phantom extra class.

This cell regenerates the list, compares it to what is already on disk, and only writes
when something actually differs.

In [ ]:
from torchvision.datasets import ImageFolder

train_dir = BIRDS_DIR / "train"
classes = sorted(d.name for d in train_dir.iterdir() if d.is_dir())

# the order torch will actually assign at training time
ds = ImageFolder(str(train_dir))
assert ds.classes == classes, "sorted() disagrees with ImageFolder ordering"
print(f"{len(classes)} classes, e.g. {classes[:3]}")

label_file = DATA_DIR / "birds525.txt"
new_text = "\n".join(c.lower() for c in classes)          # no trailing newline

if label_file.exists():
    old_text = label_file.read_text()
    if old_text == new_text:
        print(f"\n{label_file} already correct ({len(new_text.split(chr(10)))} entries) -- unchanged")
    else:
        old_lines, new_lines = old_text.split("\n"), new_text.split("\n")
        print(f"\nMISMATCH with existing {label_file}")
        print(f"  existing: {len(old_lines)} entries")
        print(f"  from dirs: {len(new_lines)} entries")
        print(f"  only in existing: {sorted(set(old_lines) - set(new_lines))[:5]}")
        print(f"  only on disk:     {sorted(set(new_lines) - set(old_lines))[:5]}")
        label_file.write_text(new_text)
        print(f"  -> overwrote with the directory-derived list")
else:
    label_file.write_text(new_text)
    print(f"\nwrote {label_file}")

## 4. Summary

In [ ]:
print("BioCLIP")
for f in sorted(BIOCLIP_DIR.iterdir()):
    print(f"  {f.name:34s} {f.stat().st_size / 1e6:8.1f} MB")

print("\nBIRDS 525")
total = 0
for split in ("train", "val", "test"):
    d = BIRDS_DIR / split
    if not d.exists():
        print(f"  {split:6s} MISSING")
        continue
    n_img = sum(1 for _ in d.rglob("*.jpg"))
    n_cls = sum(1 for x in d.iterdir() if x.is_dir())
    total += n_img
    print(f"  {split:6s} {n_cls:4d} classes  {n_img:7d} images")
print(f"  {'total':6s} {'':4s}          {total:7d} images")

print(f"\nconcept set: {DATA_DIR / 'concept_sets' / 'birds525_filtered.txt'}")
print(f"class list:  {DATA_DIR / 'birds525.txt'}")
print("\nReady to train -- see README.md 'Training' for the command.")